# 模块5 第1次课：语音增强基础 + DeepFilterNet论文精读本notebook包含：1. 语音增强问题定义与信号模型2. 传统语音增强方法回顾（谱减法、维纳滤波）3. 评估指标详解（PESQ、STOI、SI-SDR）4. DeepFilterNet论文结构化阅读引导---

## §1 语音增强问题定义**信号模型**：- x(t) = s(t) + n(t)- x(t)：带噪语音（观察信号）- s(t)：干净语音（目标信号）- n(t)：噪声（需要去除的干扰）**语音增强的目标**：从 x(t) 中估计出 s(t)，即最小化 |s_hat(t) - s(t)|。在CI场景中：- x(t) = CI麦克风拾取的带噪语音- s(t) = 理想情况下CI用户应该接收到的干净语音- 语音增强是前端处理，ACE策略是后端编码

In [ ]:
import numpy as npimport matplotlib.pyplot as plt# 演示：生成带噪语音sr = 16000duration = 3.0t = np.linspace(0, duration, int(sr * duration), endpoint=False)# 生成干净语音（多谐波）f0 = 150clean = np.zeros_like(t)for h in range(1, 10):    clean += (0.5 / h) * np.sin(2 * np.pi * f0 * h * t)clean = clean / np.max(np.abs(clean)) * 0.8# 生成噪声np.random.seed(42)noise = np.random.randn(len(t)) * 0.3# 按指定SNR混合def mix_at_snr(clean, noise, snr_db):    clean_power = np.mean(clean ** 2)    noise_power = np.mean(noise ** 2)    snr_linear = 10 ** (snr_db / 10)    noise_scaled = noise * np.sqrt(clean_power / (noise_power * snr_linear + 1e-8))    return clean + noise_scalednoisy_0db = mix_at_snr(clean, noise, 0)noisy_5db = mix_at_snr(clean, noise, 5)noisy_10db = mix_at_snr(clean, noise, 10)print("信号形状:", clean.shape)print("干净语音能量: %.6f" % np.mean(clean**2))

## §2 传统语音增强方法### 2.1 谱减法 (Spectral Subtraction)最简单的语音增强方法：- |S_hat(f)|^2 = |X(f)|^2 - |N_hat(f)|^2- 缺点：会产生"音乐噪声"（频谱上的随机亮点）### 2.2 维纳滤波 (Wiener Filter)最优线性滤波器：- H(f) = P_ss(f) / (P_ss(f) + P_nn(f))- P_ss(f)：干净语音的功率谱- P_nn(f)：噪声的功率谱- H(f) 是频率相关的增益函数维纳滤波的局限：假设信号和噪声是不相关的，且需要估计功率谱

In [ ]:
# 演示谱减法和维纳滤波from scipy.fft import fft, ifftN = len(clean)clean_fft = fft(clean)noise_fft = fft(noise)noisy_fft = fft(noisy_0db)# 谱减法noise_power = np.abs(noise_fft) ** 2clean_power = np.abs(clean_fft) ** 2estimated_power = np.abs(noisy_fft) ** 2 - noise_powerestimated_power = np.maximum(estimated_power, 0)estimated_fft = np.sqrt(estimated_power) * np.exp(1j * np.angle(noisy_fft))enhanced_spectral = np.real(ifft(estimated_fft))# 维纳滤波wiener_gain = clean_power / (clean_power + noise_power + 1e-8)enhanced_wiener = np.real(ifft(wiener_gain * noisy_fft))# 可视化对比fig, axes = plt.subplots(4, 1, figsize=(12, 8))axes[0].plot(t[:1600], clean[:1600])axes[0].set_title("干净语音")axes[1].plot(t[:1600], noisy_0db[:1600])axes[1].set_title("带噪语音 (0 dB SNR)")axes[2].plot(t[:1600], enhanced_spectral[:1600])axes[2].set_title("谱减法增强")axes[3].plot(t[:1600], enhanced_wiener[:1600])axes[3].set_title("维纳滤波增强")for ax in axes:    ax.set_xlabel("时间 (s)")    ax.set_ylabel("幅度")plt.tight_layout()plt.show()

## §3 语音增强评估指标语音增强领域有三个核心客观指标：| 指标 | 全称 | 衡量什么 | 范围 | 与CI的关系 ||------|------|---------|------|------------|| PESQ | Perceptual Evaluation of Speech Quality | 感知语音质量 | -0.5 ~ 4.5 | CI用户对音质的感知 || STOI | Short-Time Objective Intelligibility | 短时语音可懂度 | 0 ~ 1 | CI用户能听懂多少 || SI-SDR | Scale-Invariant Signal-to-Distortion Ratio | 信号失真比 | -inf ~ +inf dB | 整体增强质量 |> **关键洞察**：CI用户关心的是**可懂度**（STOI），而不仅仅是**音质**（PESQ）。语音增强不能只让声音好听，必须让CI用户能听懂。

In [ ]:
# 计算评估指标def compute_si_sdr(estimated, reference):    """计算 SI-SDR (Scale-Invariant SDR)"""    reference = reference - np.mean(reference)    estimated = estimated - np.mean(estimated)    alpha = np.dot(estimated, reference) / (np.dot(reference, reference) + 1e-8)    s_target = alpha * reference    e_noise = estimated - s_target    si_sdr = 10 * np.log10(np.dot(s_target, s_target) / (np.dot(e_noise, e_noise) + 1e-8))    return si_sdr# 评估谱减法和维纳滤波methods = {    "谱减法": enhanced_spectral,    "维纳滤波": enhanced_wiener,}print("评估指标对比 (0 dB SNR 输入):")print("-" * 50)for name, enhanced in methods.items():    si_sdr = compute_si_sdr(enhanced, clean)    print("  %s: SI-SDR = %.1f dB" % (name, si_sdr))si_sdr_input = compute_si_sdr(noisy_0db, clean)print()print("输入 (0dB SNR): SI-SDR = %.1f dB" % si_sdr_input)print()print("完整评估需要安装: pip install pesq pystoi")

## §4 指标深入理解### PESQ- 基于人耳感知模型，模拟主观MOS评分- 范围：-0.5 到 4.5（越高越好）- 主要衡量**音质**，不一定反映可懂度### STOI- 基于短时频谱的相关性- 范围：0 到 1（越高越好）- 主要衡量**可懂度**，与语音识别准确率高度相关- **对CI尤其重要**：CI用户最关心的是能不能听懂### SI-SDR- 信号层面的失真度量- 单位：dB（越高越好）- 不考虑感知，纯粹从信号角度评估- 优点：可微分，可作为训练损失函数> **思考题**：为什么仅提高SI-SDR不一定能提高STOI？试从频谱角度分析。

## §5 DeepFilterNet 论文精读### §5.1 论文阅读引导卡**第一遍：标题和摘要（5分钟）**- [ ] DeepFilterNet 要解决什么问题？- [ ] "Deep Filtering"是什么？与传统滤波有什么区别？- [ ] 为什么选择 ERB（等效矩形带宽）域而非梅尔域？**第二遍：引言（10分钟）**- [ ] 传统语音增强方法有什么局限？- [ ] 为什么深度学习比传统方法更适合语音增强？- [ ] DeepFilterNet 的核心创新点是什么？**第三遍：方法（20分钟）**- [ ] 为什么用两阶段（Enhancement + Deep Filtering）？- [ ] ERB 域与 STFT 域的区别是什么？- [ ] Deep Filtering 的数学公式是什么？- [ ] 损失函数包含哪些项？**第四遍：实验（10分钟）**- [ ] 训练数据是什么？如何构造带噪-干净配对？- [ ] 评估用了哪些数据集和指标？- [ ] 与其他方法对比如何？

### §5.2 DeepFilterNet 核心架构DeepFilterNet 的两阶段设计：**阶段1：Enhancement（增强）**- 输入: 带噪语谱图 (STFT)- ERB特征提取 -> Encoder(2D Conv) -> GroupedGRU -> ErbDecoder- 输出: ERB域掩码 -> 应用于语谱图 -> 预增强语谱图**阶段2：Deep Filtering（深度滤波）**- 预增强语谱图- Encoder中的DF分支 -> DfDecoder(GroupedGRU) -> DF系数- DF系数对低频部分做时域滤波: Y[f] = sum_k(C[k] * X[f-k])- alpha 控制DF的混合比例> 关键理解：ERB（Equivalent Rectangular Bandwidth）模拟了人耳的频率分辨率，与CI的电极映射有内在联系。

### §5.3 关键创新点与已有知识对照| DeepFilterNet组件 | 对应已有知识 | 创新之处 ||------------------|------------|----------|| ERB特征提取 | 模块3的梅尔频谱图 | ERB刻度模拟人耳频率分辨率 || Enhancement阶段 | 模块2的语谱图分类 | 输出掩码而非类别 || Deep Filtering | 传统FIR滤波 | 用网络学习滤波器系数 || GroupedGRU | 模块3的GRU | 将GRU分成多组并行，减少参数 || 两阶段设计 | — | 先粗增强再精细修复 || alpha参数 | — | 控制DF阶段参与程度，自适应切换 |

### §5.4 ERB vs 梅尔刻度- 梅尔刻度：基于纯音感知，常用于ASR和声音分类- ERB刻度：基于噪声带感知，更接近人耳实际频率分辨率- CI电极映射：22个电极约等于22个ERB频带- DeepFilterNet：32个ERB频带（更精细但仍与CI兼容）> ERB刻度与CI的关系比梅尔刻度更直接——CI的电极频率分配本身就遵循类似ERB的非线性映射。

## §6 讨论与思考题1. **方法论思考**：传统谱减法、维纳滤波用固定规则（统计模型）做增强，而深度学习用数据驱动。两种范式各有什么优劣？2. **指标选择**：如果你要评估一个语音增强系统对CI用户的帮助，你会优先看PESQ还是STOI？为什么？3. **Deep Filtering**：为什么DF只对低频部分操作？这与语音信号的特性有什么关系？4. **与ACE的关系**：如果将DeepFilterNet放在ACE之前（先增强再编码），这对ACE的通道选择会有什么影响？5. **实时性**：DeepFilterNet设计为实时运行（算法延迟5ms）。为什么实时性对CI/助听器场景至关重要？

---## 小结本节课我们：1. 了解了语音增强的信号模型和问题定义2. 实现了传统方法（谱减法、维纳滤波）并观察其局限3. 深入理解了三个核心评估指标（PESQ、STOI、SI-SDR）4. 按结构化方法精读了DeepFilterNet论文5. 建立了ERB域与CI电极映射的联系**课后任务**：- 完成论文阅读引导卡中的所有问题- 画出DeepFilterNet的完整数据流图- 预习下次课：阅读 DeepFilterNet/df/ 目录下的代码文件